# NSGA-II Multi-Objective Optimization for IDS Feature Selection & Hyperparameter Tuning

This notebook implements NSGA-II (Non-dominated Sorting Genetic Algorithm II) for simultaneously optimizing:
1. **Maximize** detection accuracy (F1-score)
2. **Minimize** false positive rate
3. **Minimize** the number of features selected

**Dataset:** CICIDS2017 (cleaned, SMOTE-balanced version from Dataset_Cleaned/)  
**Base Model:** Random Forest (consistent with team's baseline)  
**Dependencies:** `pip install numpy pandas scikit-learn matplotlib deap`

## Section 1: Imports

In [ ]:
import numpy as np
import pandas as pd
import os
import time
import random
import warnings
import json
from datetime import datetime

# Scikit-learn
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)

# DEAP - for NSGA-II
from deap import base, creator, tools, algorithms

# Visualization
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')

print("All libraries imported successfully!")
print(f"Run started at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

## Section 2: Load the Dataset

In [ ]:
# Path to the cleaned dataset folder
DATASET_DIR = "Dataset_Cleaned"

print("Loading dataset files...")
print(f"Looking in: {os.path.abspath(DATASET_DIR)}\n")

# Load the pre-split, SMOTE-balanced training data and test data
X_train = pd.read_csv(os.path.join(DATASET_DIR, "X_train_smote.csv"))
y_train = pd.read_csv(os.path.join(DATASET_DIR, "y_train_smote.csv"))
X_test  = pd.read_csv(os.path.join(DATASET_DIR, "X_test.csv"))
y_test  = pd.read_csv(os.path.join(DATASET_DIR, "y_test.csv"))

# Flatten y from DataFrame to 1D array
y_train = y_train["Label"].values
y_test  = y_test["Label"].values

# Store feature names before converting to numpy
FEATURE_NAMES = list(X_train.columns)
NUM_FEATURES  = len(FEATURE_NAMES)

# Convert X to numpy arrays for faster indexing
X_train = X_train.values
X_test  = X_test.values

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape:  {X_test.shape}")
print(f"y_train shape: {y_train.shape}  (0s: {np.sum(y_train==0)}, 1s: {np.sum(y_train==1)})")
print(f"y_test shape:  {y_test.shape}   (0s: {np.sum(y_test==0)}, 1s: {np.sum(y_test==1)})")
print(f"Number of features: {NUM_FEATURES}")
print(f"Feature names: {FEATURE_NAMES[:5]}... (showing first 5)")

## Section 3: Define Hyperparameter Bounds and Individual Encoding

In [ ]:
# Random Forest hyperparameter bounds for NSGA-II to search
HP_BOUNDS = {
    'n_estimators':     (10, 200),    # Number of trees in the forest
    'max_depth':        (3, 30),      # Maximum depth of each tree
    'min_samples_split': (2, 20),     # Min samples needed to split a node
    'min_samples_leaf':  (1, 10),     # Min samples in a leaf node
}

# Each individual = binary feature mask (45) + 4 hyperparameters = 49 genes
IND_SIZE = NUM_FEATURES + 4
print(f"Individual size: {IND_SIZE} ({NUM_FEATURES} features + 4 hyperparams)")

## Section 4: Decode and Evaluate Functions

In [ ]:
def decode_individual(individual):
    """
    Convert a list of floats into a feature mask and hyperparameters.
    
    The individual is a list of numbers between 0 and 1.
    - First NUM_FEATURES values: if > 0.5, that feature is selected
    - Last 4 values: scaled to hyperparameter ranges
    """
    # Feature selection: threshold at 0.5
    feature_mask = np.array(individual[:NUM_FEATURES]) > 0.5
    
    # Make sure at least 1 feature is selected
    if not np.any(feature_mask):
        feature_mask[np.random.randint(0, NUM_FEATURES)] = True
    
    # Decode hyperparameters (scale from [0,1] to actual ranges)
    hp = individual[NUM_FEATURES:]
    hp = [max(0.0, min(1.0, g)) for g in hp]  # clamp to [0, 1]
    
    hyperparams = {
        'n_estimators':     int(hp[0] * 190 + 10),   # 10 to 200
        'max_depth':        int(hp[1] * 27 + 3),      # 3 to 30
        'min_samples_split': int(hp[2] * 18 + 2),     # 2 to 20
        'min_samples_leaf':  int(hp[3] * 9 + 1),      # 1 to 10
    }
    
    return feature_mask, hyperparams


def evaluate(individual):
    """
    FITNESS FUNCTION - called by NSGA-II for every candidate solution.
    
    Trains a Random Forest with the selected features and hyperparameters,
    then returns 3 objectives (all to be MINIMIZED):
      1. Negative F1-score (minimizing negative = maximizing F1)
      2. False Positive Rate
      3. Feature fraction (proportion of features used)
    """
    feature_mask, hyperparams = decode_individual(individual)
    
    # Select only the chosen features
    X_tr = X_train[:, feature_mask]
    X_te = X_test[:, feature_mask]
    num_selected = int(np.sum(feature_mask))
    
    try:
        # Build and train the Random Forest
        clf = RandomForestClassifier(
            n_estimators=hyperparams['n_estimators'],
            max_depth=hyperparams['max_depth'],
            min_samples_split=hyperparams['min_samples_split'],
            min_samples_leaf=hyperparams['min_samples_leaf'],
            random_state=42,
            n_jobs=-1  # use all CPU cores
        )
        clf.fit(X_tr, y_train)
        y_pred = clf.predict(X_te)
        
        # Calculate objectives
        f1 = f1_score(y_test, y_pred, average='weighted')
        
        cm = confusion_matrix(y_test, y_pred)
        if cm.shape == (2, 2):
            tn, fp, fn, tp = cm.ravel()
            fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0
        else:
            fpr = 0.0
        
        feature_fraction = num_selected / NUM_FEATURES
        
    except Exception as e:
        print(f"  [ERROR] {e}")
        return (0.0, 1.0, 1.0)  # worst case values
    
    # All three objectives are MINIMIZED
    # Negate F1 so minimizing it = maximizing actual F1
    return (-f1, fpr, feature_fraction)

print("Evaluate and decode functions defined.")

## Section 5: Setup NSGA-II

In [ ]:
# Clean up any previous DEAP definitions (needed if you re-run)
if hasattr(creator, "FitnessMulti"):
    del creator.FitnessMulti
if hasattr(creator, "Individual"):
    del creator.Individual

# 3 objectives, all minimized (weights are all -1)
creator.create("FitnessMulti", base.Fitness, weights=(-1.0, -1.0, -1.0))
creator.create("Individual", list, fitness=creator.FitnessMulti)

toolbox = base.Toolbox()
toolbox.register("attr_float", random.random)
toolbox.register("individual", tools.initRepeat, creator.Individual,
                 toolbox.attr_float, n=IND_SIZE)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)
toolbox.register("evaluate", evaluate)

# Crossover: Simulated Binary Crossover (good for real-valued genes)
toolbox.register("mate", tools.cxSimulatedBinaryBounded,
                 low=0.0, up=1.0, eta=20.0)

# Mutation: Polynomial Bounded Mutation
toolbox.register("mutate", tools.mutPolynomialBounded,
                 low=0.0, up=1.0, eta=20.0, indpb=1.0/IND_SIZE)

# Selection: NSGA-II non-dominated sorting
toolbox.register("select", tools.selNSGA2)

print("NSGA-II configured successfully!")

## Section 6: Run the Optimization

In [ ]:
# NSGA-II Parameters
POPULATION_SIZE = 50
N_GENERATIONS   = 30
CROSSOVER_PROB  = 0.8
MUTATION_PROB   = 0.2

print(f"Population size:  {POPULATION_SIZE}")
print(f"Generations:      {N_GENERATIONS}")
print(f"Crossover prob:   {CROSSOVER_PROB}")
print(f"Mutation prob:    {MUTATION_PROB}")

# Set seeds for reproducibility
random.seed(42)
np.random.seed(42)

# Create initial population
population = toolbox.population(n=POPULATION_SIZE)

# Track statistics per generation
stats = tools.Statistics(lambda ind: ind.fitness.values)
stats.register("min", np.min, axis=0)
stats.register("max", np.max, axis=0)
stats.register("avg", np.mean, axis=0)

# Pareto front tracker
pareto_front = tools.ParetoFront()

# Run NSGA-II
start_time = time.time()

population, logbook = algorithms.eaMuPlusLambda(
    population,
    toolbox,
    mu=POPULATION_SIZE,
    lambda_=POPULATION_SIZE,
    cxpb=CROSSOVER_PROB,
    mutpb=MUTATION_PROB,
    ngen=N_GENERATIONS,
    stats=stats,
    halloffame=pareto_front,
    verbose=True
)

end_time = time.time()
total_time = end_time - start_time

print(f"\nOptimization Complete!")
print(f"Runtime: {total_time:.1f} seconds ({total_time/60:.1f} minutes)")
print(f"Pareto front: {len(pareto_front)} non-dominated solutions")

## Section 7: Analyse Pareto Front

In [ ]:
results = []
for i, ind in enumerate(pareto_front):
    neg_f1, fpr, feat_frac = ind.fitness.values
    feature_mask, hyperparams = decode_individual(ind)
    num_features = int(np.sum(feature_mask))
    
    results.append({
        'solution_id': i + 1,
        'f1_score': round(-neg_f1, 4),
        'false_positive_rate': round(fpr, 4),
        'num_features': num_features,
        'feature_fraction': round(feat_frac, 4),
        'n_estimators': hyperparams['n_estimators'],
        'max_depth': hyperparams['max_depth'],
        'min_samples_split': hyperparams['min_samples_split'],
        'min_samples_leaf': hyperparams['min_samples_leaf'],
    })

results_df = pd.DataFrame(results)

print("PARETO FRONT SOLUTIONS:")
print(results_df.to_string(index=False))

## Section 8: Select Best Compromise Solution

In [ ]:
# Pick the solution closest to the ideal point
# Ideal = highest F1, lowest FPR, fewest features

df_n = results_df.copy()
for col in ['f1_score', 'false_positive_rate', 'num_features']:
    cmin, cmax = df_n[col].min(), df_n[col].max()
    df_n[col + '_n'] = (df_n[col] - cmin) / (cmax - cmin) if cmax > cmin else 0.0

df_n['distance'] = np.sqrt(
    (1 - df_n['f1_score_n'])**2 +
    (df_n['false_positive_rate_n'])**2 +
    (df_n['num_features_n'])**2
)

best_idx = df_n['distance'].idxmin()
best = results_df.iloc[best_idx]

print("BEST COMPROMISE SOLUTION")
print(f"  F1-Score:            {best['f1_score']}")
print(f"  False Positive Rate: {best['false_positive_rate']}")
print(f"  Features Selected:   {int(best['num_features'])}/{NUM_FEATURES}")
print(f"  n_estimators:        {int(best['n_estimators'])}")
print(f"  max_depth:           {int(best['max_depth'])}")
print(f"  min_samples_split:   {int(best['min_samples_split'])}")
print(f"  min_samples_leaf:    {int(best['min_samples_leaf'])}")

## Section 9: Detailed Evaluation of Best Solution

In [ ]:
best_ind = pareto_front[best_idx]
feature_mask, hyperparams = decode_individual(best_ind)

X_tr_best = X_train[:, feature_mask]
X_te_best = X_test[:, feature_mask]

best_clf = RandomForestClassifier(
    n_estimators=hyperparams['n_estimators'],
    max_depth=hyperparams['max_depth'],
    min_samples_split=hyperparams['min_samples_split'],
    min_samples_leaf=hyperparams['min_samples_leaf'],
    random_state=42,
    n_jobs=-1
)
best_clf.fit(X_tr_best, y_train)
y_pred = best_clf.predict(X_te_best)

# Full metrics
accuracy  = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='weighted')
recall    = recall_score(y_test, y_pred, average='weighted')
f1        = f1_score(y_test, y_pred, average='weighted')

cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()
detection_rate = tp / (tp + fn) if (tp + fn) > 0 else 0
fpr_final = fp / (fp + tn) if (fp + tn) > 0 else 0

selected_features = [FEATURE_NAMES[i] for i in range(NUM_FEATURES) if feature_mask[i]]

print("DETAILED CLASSIFICATION REPORT")
print(classification_report(y_test, y_pred, target_names=['Normal', 'Attack']))
print(f"Confusion Matrix:")
print(f"  TN={tn}  FP={fp}")
print(f"  FN={fn}  TP={tp}")
print(f"\nFinal Metrics:")
print(f"  Accuracy:            {accuracy:.4f}")
print(f"  Precision (weighted):{precision:.4f}")
print(f"  Recall (weighted):   {recall:.4f}")
print(f"  F1-Score (weighted): {f1:.4f}")
print(f"  Detection Rate (TPR):{detection_rate:.4f}")
print(f"  False Positive Rate: {fpr_final:.4f}")
print(f"  Features Selected:   {len(selected_features)}/{NUM_FEATURES}")
print(f"  Runtime:             {total_time:.1f} seconds")
print(f"\nSelected Features:")
for feat in selected_features:
    print(f"  - {feat}")

## Section 10: Visualizations

In [ ]:
# Plot 1: F1-Score vs False Positive Rate
fig, ax = plt.subplots(figsize=(8, 6))
sc = ax.scatter(
    results_df['false_positive_rate'], results_df['f1_score'],
    c=results_df['num_features'], cmap='viridis',
    s=60, edgecolors='black', linewidths=0.5
)
ax.scatter(best['false_positive_rate'], best['f1_score'],
           c='red', s=200, marker='*', zorder=5, label='Best Compromise')
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('F1-Score', fontsize=12)
ax.set_title('NSGA-II Pareto Front: F1-Score vs FPR', fontsize=14)
plt.colorbar(sc, label='Number of Features')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('outputs/nsga2/pareto_f1_vs_fpr.png', dpi=300)
plt.show()

In [ ]:
# Plot 2: F1-Score vs Feature Count
fig, ax = plt.subplots(figsize=(8, 6))
sc = ax.scatter(
    results_df['num_features'], results_df['f1_score'],
    c=results_df['false_positive_rate'], cmap='RdYlGn_r',
    s=60, edgecolors='black', linewidths=0.5
)
ax.scatter(best['num_features'], best['f1_score'],
           c='red', s=200, marker='*', zorder=5, label='Best Compromise')
ax.set_xlabel('Number of Features Selected', fontsize=12)
ax.set_ylabel('F1-Score', fontsize=12)
ax.set_title('NSGA-II Pareto Front: F1-Score vs Feature Count', fontsize=14)
plt.colorbar(sc, label='False Positive Rate')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('outputs/nsga2/pareto_f1_vs_features.png', dpi=300)
plt.show()

In [ ]:
# Plot 3: 3D Pareto Front
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')
ax.scatter(results_df['false_positive_rate'], results_df['num_features'],
           results_df['f1_score'], c='steelblue', s=50, alpha=0.7)
ax.scatter([best['false_positive_rate']], [best['num_features']],
           [best['f1_score']], c='red', s=200, marker='*')
ax.set_xlabel('FPR')
ax.set_ylabel('Num Features')
ax.set_zlabel('F1-Score')
ax.set_title('NSGA-II 3D Pareto Front', fontsize=14)
plt.tight_layout()
plt.savefig('outputs/nsga2/pareto_3d.png', dpi=300)
plt.show()

In [ ]:
# Plot 4: Convergence over generations
gen = logbook.select("gen")
avg_fitness = logbook.select("avg")
avg_f1   = [-x[0] for x in avg_fitness]
avg_fpr  = [x[1] for x in avg_fitness]
avg_feat = [x[2] for x in avg_fitness]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
axes[0].plot(gen, avg_f1, 'b-', linewidth=2)
axes[0].set_xlabel('Generation'); axes[0].set_ylabel('Avg F1-Score')
axes[0].set_title('F1-Score Convergence'); axes[0].grid(True, alpha=0.3)

axes[1].plot(gen, avg_fpr, 'r-', linewidth=2)
axes[1].set_xlabel('Generation'); axes[1].set_ylabel('Avg FPR')
axes[1].set_title('FPR Convergence'); axes[1].grid(True, alpha=0.3)

axes[2].plot(gen, avg_feat, 'g-', linewidth=2)
axes[2].set_xlabel('Generation'); axes[2].set_ylabel('Avg Feature Fraction')
axes[2].set_title('Feature Reduction'); axes[2].grid(True, alpha=0.3)

plt.suptitle('NSGA-II Convergence Over Generations', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('outputs/nsga2/convergence.png', dpi=300, bbox_inches='tight')
plt.show()

## Section 11: Save Results

In [ ]:
os.makedirs("outputs/nsga2", exist_ok=True)

# Save Pareto front results
results_df.to_csv('outputs/nsga2/pareto_front_results.csv', index=False)
print("Saved: outputs/nsga2/pareto_front_results.csv")

# Save best solution details
best_results = {
    'algorithm': 'NSGA-II',
    'accuracy': float(accuracy),
    'precision': float(precision),
    'recall': float(recall),
    'f1_score': float(f1),
    'detection_rate': float(detection_rate),
    'false_positive_rate': float(fpr_final),
    'num_features_selected': len(selected_features),
    'total_features': NUM_FEATURES,
    'feature_reduction_pct': round(1 - len(selected_features)/NUM_FEATURES, 4),
    'selected_features': selected_features,
    'hyperparameters': {k: int(v) for k, v in hyperparams.items()},
    'runtime_seconds': round(total_time, 2),
    'nsga2_params': {
        'population_size': POPULATION_SIZE,
        'n_generations': N_GENERATIONS,
        'crossover_prob': CROSSOVER_PROB,
        'mutation_prob': MUTATION_PROB,
    },
    'pareto_front_size': len(pareto_front),
}
with open('outputs/nsga2/best_solution.json', 'w') as f:
    json.dump(best_results, f, indent=2)
print("Saved: outputs/nsga2/best_solution.json")

# Save selected feature names
with open('outputs/nsga2/selected_features.txt', 'w') as f:
    for feat in selected_features:
        f.write(feat + '\n')
print("Saved: outputs/nsga2/selected_features.txt")

## Section 12: Comparison Table

In [ ]:
print(f"{'Method':<20} {'Accuracy':<12} {'F1-Score':<12} {'Precision':<12} "
      f"{'Recall':<12} {'FPR':<12} {'Features':<12} {'Time(s)':<12}")
print("-" * 100)
print(f"{'Baseline (RF)':<20} {'---':<12} {'---':<12} {'---':<12} "
      f"{'---':<12} {'---':<12} {str(NUM_FEATURES):<12} {'---':<12}")
print(f"{'GA-optimized':<20} {'---':<12} {'---':<12} {'---':<12} "
      f"{'---':<12} {'---':<12} {'---':<12} {'---':<12}")
print(f"{'PSO-optimized':<20} {'---':<12} {'---':<12} {'---':<12} "
      f"{'---':<12} {'---':<12} {'---':<12} {'---':<12}")
print(f"{'SA-optimized':<20} {'---':<12} {'---':<12} {'---':<12} "
      f"{'---':<12} {'---':<12} {'---':<12} {'---':<12}")
print(f"{'NSGA-II (best)':<20} {accuracy:<12.4f} {f1:<12.4f} {precision:<12.4f} "
      f"{recall:<12.4f} {fpr_final:<12.4f} "
      f"{str(len(selected_features)):<12} {total_time:<12.1f}")
print("-" * 100)
print("(Replace '---' with your teammates' results)")

print(f"\nCompleted at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")